# Drug-disease repurposing

💡 **Environment:** `clamp-analyses`

Compares the three canonical compendia (ARCHS4, recount2, GTEx) and the gene-space
baseline (no CLAMP model) head to head on the same drug-repurposing task as `00`/`01`:
bootstrap dominance probability P(row > column) over the max-aggregated AUROC, and a
per-tissue AUROC boxplot across all 49 GTEx tissues ordered by descending
performance (diamonds mark the max-over-tissue reference score, brackets show the
Wilcoxon signed-rank test of ARCHS4 against each other method, BH-corrected).

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import yaml
from pyprojroot import here
import matplotlib.pyplot as plt

with open(here('config.yaml'), 'r') as f:
    colors = yaml.safe_load(f)['DRUG_DISEASE_MODEL_COLORS']

DATA = Path(snakemake.input.stats).parent
print((DATA / 'figure_statistics.txt').read_text())
dominance = pd.read_csv(DATA / 'ordering_stability.csv')
tissue = pd.read_csv(DATA / 'per_tissue_metrics.csv')
aggregate = pd.read_csv(DATA / 'max_aggregate_reference.csv')
tests = pd.read_csv(DATA / 'paired_tissue_tests.csv')
order = aggregate.set_index('method')['auroc'].sort_values(ascending=False).index.tolist()
n = len(order)
assert tissue.tissue.nunique() == 49 and set(tissue.method) == set(order)

SUPERSCRIPT = str.maketrans('0123456789-', '⁰¹²³⁴⁵⁶⁷⁸⁹⁻')


def fmt_p(x):
    if x <= 0:
        return '0'
    mantissa, exp = f'{x:.2e}'.split('e')
    return f'{mantissa}×10{str(int(exp)).translate(SUPERSCRIPT)}'


fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1, 1.25]})
matrix = dominance.pivot(index='row_m', columns='col_m', values='value').reindex(index=order, columns=order)
matrix = matrix.mask(np.eye(n, dtype=bool))
im = ax0.imshow(matrix, vmin=0, vmax=1, cmap='BuGn')
for i in range(n):
    for j in range(n):
        if pd.notna(matrix.iloc[i, j]):
            v = matrix.iloc[i, j]
            ax0.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=15, weight='bold', color='white' if v >= .7 else 'black')
ax0.set(xticks=range(n), xticklabels=order, yticks=range(n), yticklabels=order, title='Bootstrap dominance probability (AUROC)')
fig.colorbar(im, ax=ax0, label='P(row > column)')
values = [tissue.loc[tissue.method == method, 'auroc'].to_numpy() for method in order]
box = ax1.boxplot(values, labels=order, patch_artist=True, showfliers=False)
for patch, method in zip(box['boxes'], order): patch.set(facecolor=colors[method], alpha=.55)
rng = np.random.default_rng(42)
for i, (method, vals) in enumerate(zip(order, values), 1): ax1.scatter(rng.normal(i, .055, len(vals)), vals, color=colors[method], alpha=.55, s=26)
for i, method in enumerate(order, 1):
    score = aggregate.loc[aggregate.method == method, 'auroc'].iloc[0]
    ax1.scatter(i, score, marker='D', s=135, color='white', edgecolor='black', zorder=4)
top = max(tissue.auroc.max(), aggregate.auroc.max())
xpos = {method: i for i, method in enumerate(order, 1)}
archs4_tests = tests[(tests.group1 == 'ARCHS4') | (tests.group2 == 'ARCHS4')].reset_index(drop=True)
for i, row in archs4_tests.iterrows():
    x1, x2, y = xpos[row.group1], xpos[row.group2], top + (.012 + .014 * i)
    ax1.plot([x1, x2], [y, y], color='black')
    ax1.text((x1 + x2) / 2, y + .002, f"q = {fmt_p(row.q_value_bh)}", ha='center', fontsize=9)
ax1.set(ylabel='AUROC (per GTEx tissue)', title='49 tissues; diamonds = max-over-tissue')
ax1.set_ylim(top - .14, top + .05 + .014 * max(0, len(archs4_tests) - 2))
fig.tight_layout()
plt.show()

display(aggregate)
display(tests.assign(p_value=tests.p_value.map(fmt_p), q_value_bh=tests.q_value_bh.map(fmt_p)))